# Assistente de Voz - Python + JavaScript

Projeto DIO usando Python (processamento) + JavaScript (gravação de áudio no navegador).

Execute cada célula na ordem.

## 1. Instalar dependências (Python)

In [ ]:
# Instala bibliotecas Python necessárias
!pip install -q openai gTTS
!pip install -q git+https://github.com/openai/whisper.git

## 2. Configurar API Key (OpenAI)

In [ ]:
import os

# Cole sua API Key aqui (entre as aspas)
OPENAI_API_KEY = "sua_chave_aqui"

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print("API Key configurada!")

## 3. Gravar áudio (JavaScript no navegador)

Execute a célula abaixo e fale algo quando aparecer "Ouvindo..."

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do microfone
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = e => resolve(e.srcElement.result);
    reader.readAsDataURL(blob);
});

var record = time => new Promise(async resolve => {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    recorder = new MediaRecorder(stream);
    chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();
    await sleep(time);
    recorder.onstop = async () => {
        blob = new Blob(chunks);
        text = await b2text(blob);
        resolve(text);
    };
    recorder.stop();
});
"""

def gravar_audio(segundos=5):
    """Usa JavaScript para gravar áudio do navegador"""
    display(Javascript(RECORD_JS))
    print(f"Ouvindo... (fale por {segundos} segundos)")
    
    # Chama o JavaScript e aguarda o resultado
    js_result = output.eval_js(f'record({segundos * 1000})')
    
    # Decodifica o áudio base64
    audio = b64decode(js_result.split(',')[1])
    
    # Salva o arquivo
    arquivo = '/content/audio_gravado.wav'
    with open(arquivo, 'wb') as f:
        f.write(audio)
    
    print(f"Áudio salvo em: {arquivo}")
    return arquivo

# Executa a gravação
arquivo_audio = gravar_audio(5)

## 4. Transcrever áudio (Python + Whisper)

In [ ]:
import whisper

# Carrega o modelo Whisper
modelo = whisper.load_model("base")

# Transcreve o áudio
print("Transcrevendo...")
resultado = modelo.transcribe(arquivo_audio, language='pt')
texto_usuario = resultado["text"].strip()

print(f"Você disse: {texto_usuario}")

## 5. Processar com ChatGPT (Python + OpenAI API)

In [ ]:
import openai

openai.api_key = os.environ.get('OPENAI_API_KEY')

# Envia para o ChatGPT
print("Pensando na resposta...")
resposta = openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "Você é um assistente amigável que responde em português brasileiro de forma natural."},
        {"role": "user", "content": texto_usuario}
    ],
    max_tokens=200
)

texto_resposta = resposta.choices[0].message.content.strip()
print(f"Resposta: {texto_resposta}")

## 6. Sintetizar voz (Python + gTTS)

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

# Converte texto em áudio
print("Gerando áudio da resposta...")
tts = gTTS(text=texto_resposta, lang='pt', slow=False)

arquivo_resposta = "/content/resposta.mp3"
tts.save(arquivo_resposta)

# Reproduz o áudio
print("Tocando resposta...")
display(Audio(arquivo_resposta, autoplay=True))

## Tudo junto (versão completa em uma célula)

In [ ]:
# Executa todo o fluxo de uma vez
# Grava -> Transcreve -> Processa -> Fala

print("=== Iniciando conversa ==="")

# 1. Grava (JavaScript)
arquivo = gravar_audio(5)

# 2. Transcreve (Python + Whisper)
resultado = modelo.transcribe(arquivo, language='pt')
texto = resultado["text"].strip()
print(f"Você: {texto}")

# 3. Processa (Python + ChatGPT)
resp = openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": texto}],
    max_tokens=150
)
resposta_texto = resp.choices[0].message.content.strip()
print(f"Assistente: {resposta_texto}")

# 4. Fala (Python + gTTS)
tts = gTTS(text=resposta_texto, lang='pt', slow=False)
tts.save("/content/resposta_final.mp3")
display(Audio("/content/resposta_final.mp3", autoplay=True))

print("=== Fim da conversa ==="")

---

**Autor:** Acib ABBADE
**Bootcamp:** DIO
**Tecnologias:** Python + JavaScript + Whisper + ChatGPT + gTTS